# ============================================================
# STEP 9 — STATISTICAL SIGNIFICANCE TESTING (TUMC) - New One
# ============================================================
# Upgrades "HistGB is best" to "HistGB is SIGNIFICANTLY best".
# Reviewers comparing 6 models expect significance testing,
# not just a ranking by mean score.
#
# THREE tests:
#   1. Friedman test  — are the models significantly different
#      across folds? (omnibus, non-parametric, repeated-measures)
#   2. Nemenyi post-hoc — WHICH pairs differ? (critical-distance)
#   3. McNemar test   — best vs 2nd-best, on paired per-URL
#      predictions (the strict head-to-head)
#
# Operates on the deployment config (inductive, Exp C). Uses
# per-fold scores for Friedman/Nemenyi and out-of-fold paired
# predictions for McNemar.
#
# Output: step9_significance_report.txt, step9_cd_diagram.png
# ============================================================

In [ ]:
import os, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare, rankdata
from sklearn.ensemble import (HistGradientBoostingClassifier,
                              RandomForestClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import f1_score, matthews_corrcoef
import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings("ignore")
SEED = 42
DATA_DIR = "/content/drive/MyDrive/TUMC_dataset"
for cand in ["features_full_final_inductive_dedup.csv","features_full_v13.csv"]:
    INPUT = os.path.join(DATA_DIR, cand)
    if os.path.exists(INPUT): break
OUT = os.path.join(DATA_DIR,"step9_significance_report.txt")

TASK = "5class"          # significance most meaningful on the harder task
LEAKY = ["cluster_malicious_ratio","campaign_token_score","is_tr_domain","is_https"]
report=[]
def log(s): print(s); report.append(s)

df = pd.read_csv(INPUT, low_memory=False)
META={"url","source","tld","label","label_enc","class_final","class_enc","fold","reg_domain"}
FEATURES=[c for c in df.columns if c not in META and c not in LEAKY]
target = "class_enc" if TASK=="5class" else "label_enc"
n_classes=df[target].nunique(); is_binary=(TASK=="binary")
X=df[FEATURES].fillna(0).values; y=df[target].values; folds=df["fold"].values

def models():
    return {
    "LogisticRegression": LogisticRegression(C=1.0,max_iter=1000,
            class_weight="balanced",random_state=SEED,n_jobs=-1),
    "RandomForest": RandomForestClassifier(n_estimators=200,max_depth=12,
            min_samples_leaf=5,class_weight="balanced",random_state=SEED,n_jobs=-1),
    "HistGB": HistGradientBoostingClassifier(max_depth=6,learning_rate=0.05,
            max_iter=300,random_state=SEED),
    "XGBoost": xgb.XGBClassifier(max_depth=4,learning_rate=0.03,n_estimators=300,
            subsample=0.8,colsample_bytree=0.8,reg_alpha=1,reg_lambda=2,
            random_state=SEED,n_jobs=-1,verbosity=0,
            **({"scale_pos_weight":5.5} if is_binary else
               {"objective":"multi:softprob","num_class":n_classes})),
    "LightGBM": lgb.LGBMClassifier(num_leaves=31,max_depth=6,learning_rate=0.03,
            n_estimators=300,class_weight="balanced",random_state=SEED,
            n_jobs=-1,verbose=-1,
            **({} if is_binary else {"objective":"multiclass","num_class":n_classes})),
    }

log("="*70); log(f"STEP 9 — STATISTICAL SIGNIFICANCE ({TASK}, inductive, Exp C)"); log("="*70)

# ── Per-fold scores + stored OOF predictions for McNemar ─────
fold_scores={m:[] for m in models()}
oof_pred={m:np.zeros(len(df),dtype=int) for m in models()}
metric = "f1_macro" if not is_binary else "mcc"
log(f"\n  Collecting per-fold {metric} for {len(models())} models...")
for fid in sorted(set(folds)):
    tri=np.where(folds!=fid)[0]; tei=np.where(folds==fid)[0]
    Xmin = X.copy()
    if np.isnan(Xmin).any(): Xmin=np.nan_to_num(Xmin)
    for name,mdl in models().items():
        mdl.fit(Xmin[tri],y[tri])
        pred=mdl.predict(Xmin[tei]); oof_pred[name][tei]=pred
        s=(f1_score(y[tei],pred,average="macro",zero_division=0) if not is_binary
           else matthews_corrcoef(y[tei],pred))
        fold_scores[name].append(s)
    log(f"    fold {fid} done")

score_df=pd.DataFrame(fold_scores)
log("\n  Per-fold scores:"); log(score_df.round(4).to_string())
means=score_df.mean().sort_values(ascending=False)
log("\n  Mean ranking:")
for m,v in means.items(): log(f"    {m:<14s}: {v:.4f}")

# ── 1. Friedman ──────────────────────────────────────────────
log("\n[1] Friedman test (are models different across folds?)")
stat,p=friedmanchisquare(*[score_df[m].values for m in score_df.columns])
log(f"    chi^2={stat:.4f}  p={p:.4g}")
log(f"    → {'SIGNIFICANT differences among models (reject H0)' if p<0.05 else 'no significant difference'}")

# ── 2. Nemenyi critical distance ─────────────────────────────
log("\n[2] Nemenyi post-hoc (which models differ?)")
ranks=np.array([rankdata(-score_df.iloc[i].values) for i in range(len(score_df))])
avg_ranks=ranks.mean(0)
k=score_df.shape[1]; N=score_df.shape[0]
q_alpha={2:1.960,3:2.343,4:2.569,5:2.728,6:2.850,7:2.949}.get(k,2.850)
CD=q_alpha*np.sqrt(k*(k+1)/(6*N))
log(f"    k={k} models, N={N} folds, q_0.05={q_alpha}")
log(f"    Critical Distance (CD) = {CD:.3f}")
log("    Average ranks (lower=better):")
for m,rk in sorted(zip(score_df.columns,avg_ranks),key=lambda t:t[1]):
    log(f"      {m:<14s}: {rk:.3f}")
log("    Pairs with rank difference > CD are significantly different:")
cols=list(score_df.columns)
for i,j in itertools.combinations(range(k),2):
    d=abs(avg_ranks[i]-avg_ranks[j])
    if d>CD:
        log(f"      {cols[i]} vs {cols[j]}: diff={d:.3f} > CD ✓ significant")

# CD diagram
fig,ax=plt.subplots(figsize=(10,3))
order=np.argsort(avg_ranks)
ax.scatter(avg_ranks,[1]*k,s=60,zorder=3,color="#1D9E75")
for m,rk in zip(score_df.columns,avg_ranks):
    ax.annotate(m,(rk,1),xytext=(0,12 if list(score_df.columns).index(m)%2==0 else -18),
                textcoords="offset points",ha="center",fontsize=9)
best=avg_ranks[order[0]]
ax.plot([best,best+CD],[0.85,0.85],color="#C94F0B",lw=3)
ax.annotate(f"CD={CD:.2f}",((best+best+CD)/2,0.85),xytext=(0,-16),
            textcoords="offset points",ha="center",color="#C94F0B",fontsize=9)
ax.set_yticks([]); ax.set_xlabel("Average rank (lower = better)")
ax.set_title("Nemenyi Critical-Difference — TUMC (5-class)", fontweight="bold", pad=20)
ax.invert_xaxis();
plt.title("Nemenyi Critical-Difference — TUMC (5-class)", fontweight="bold", pad=20)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(os.path.join(DATA_DIR,"step9_cd_diagram.png"),dpi=300); plt.close()
log("    Saved CD diagram → step9_cd_diagram.png")

# ── 3. McNemar best vs 2nd ───────────────────────────────────
log("\n[3] McNemar test (best vs 2nd-best, paired per-URL)")
b1,b2=means.index[0],means.index[1]
correct1=(oof_pred[b1]==y); correct2=(oof_pred[b2]==y)
n01=int((~correct1 & correct2).sum())   # b1 wrong, b2 right
n10=int((correct1 & ~correct2).sum())   # b1 right, b2 wrong
log(f"    {b1} (best) vs {b2} (2nd)")
log(f"    {b1} right & {b2} wrong: {n10:,}")
log(f"    {b1} wrong & {b2} right: {n01:,}")
if n01+n10>0:
    # continuity-corrected McNemar chi-square
    chi=(abs(n01-n10)-1)**2/(n01+n10)
    from scipy.stats import chi2
    pmc=1-chi2.cdf(chi,1)
    log(f"    McNemar chi^2={chi:.4f}  p={pmc:.4g}")
    log(f"    → {b1} is {'SIGNIFICANTLY better than '+b2 if pmc<0.05 else 'NOT significantly different from '+b2} (p{'<0.05' if pmc<0.05 else '>=0.05'})")

log("\n"+"="*70)
log("REPORTABLE: Friedman establishes overall differences; Nemenyi CD")
log("shows which models separate; McNemar confirms the top model's edge.")
open(OUT,"w").write("\n".join(report))
print(f"\n[saved] {OUT}")

STEP 9 — STATISTICAL SIGNIFICANCE (5class, inductive, Exp C)

    fold 0 done
    fold 1 done
    fold 2 done
    fold 3 done
    fold 4 done

  Per-fold scores:
   LogisticRegression  RandomForest  HistGB  XGBoost  LightGBM
0              0.2816        0.8679  0.9113   0.8919    0.8846
1              0.2936        0.8553  0.8936   0.8732    0.8653
2              0.2897        0.8627  0.9108   0.8883    0.8836
3              0.3013        0.8598  0.9056   0.8828    0.8775
4              0.3191        0.8638  0.9023   0.8831    0.8785

  Mean ranking:
    HistGB        : 0.9047
    XGBoost       : 0.8839
    LightGBM      : 0.8779
    RandomForest  : 0.8619
    LogisticRegression: 0.2971

[1] Friedman test (are models different across folds?)
    chi^2=20.0000  p=0.0004994
    → SIGNIFICANT differences among models (reject H0)

[2] Nemenyi post-hoc (which models differ?)
    k=5 models, N=5 folds, q_0.05=2.728
    Critical Distance (CD) = 2.728
    Average ranks (lower=better):
      Hi